# Ingesta de países de producción (incremental)

Lee la carpeta `production_country` (JSON multilínea) del lote `incremental_append/<p_file_date>` en Bronze, aplica un esquema explícito, estandariza las columnas y agrega `file_date`. Carga la tabla Delta `movie_silver.production_countries` con `MERGE` (llave `movie_id` + `country_id` + `file_date`), particionada por `file_date`.

### Configuración

Carga las rutas y las funciones comunes, y lee los parámetros `p_environment` y `p_file_date` (fecha del lote).

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

### Lectura del archivo

Primero lee la carpeta `production_country` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
production_country_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_country")
    )

In [0]:
production_country_shema = StructType([
    StructField("movieId", IntegerType()),
    StructField("countryId", IntegerType())
])

production_country_df = (
    spark.read
        .schema(production_country_shema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/production_country")
)

### Transformación de columnas

Renombra `movieId` y `countryId` a snake_case y agrega `env` y `file_date`.

In [0]:
production_country_df = (
    production_country_df
        .withColumnsRenamed(
            {
                "movieId": "movie_id",
                "countryId": "country_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env),
                "file_date": lit(v_file_date)
            }
        )
)

production_country_df = add_ingestion_date(production_country_df)

### Escritura en Silver

Las celdas comentadas conservan estrategias anteriores (borrar partición + `append`). La carga activa es `merge_delta_lake` sobre `movie_silver.production_countries` con la llave `movie_id` + `country_id` + `file_date`.

In [0]:
#overwrite_partitioned_table("movie_silver", "production_countries", "file_date", v_file_date)

In [0]:
"""
(
    production_country_df
        .write
        .mode("append")
        .saveAsTable("movie_silver.production_countries")
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.country_id = src.country_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = production_country_df, schema_name = "movie_silver", table_name = "production_countries", merge_condition = merge_condition, partition_column = "file_date")

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")